In [ ]:
import os, threading, time
import numpy as np
import h5py
from scipy.interpolate import griddata
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

output_path = '/mnt/c/Users/photo/Photonics_Group/Ruihuan/kubeflow-tdgl/notebooks/sim_output.h5'
print(f'HDF5 exists: {os.path.exists(output_path)}')

# Extract mesh points directly from HDF5
with h5py.File(output_path, 'r') as f:
    points = np.array(f['solution/device/mesh/sites'])
    total = len(f['data'].keys())
    frame_times = np.array([
        float(f[f'data/{fi}'].attrs.get('time', 0))
        for fi in range(total)
    ])
print(f'Mesh: {points.shape}, Frames: {total}, Time: {frame_times[0]:.0f}-{frame_times[-1]:.0f}')

In [ ]:
je_initial, je_final, je_step = 0.0, 10.5, 0.1
ramp_time, stable_time, save_time = 50.0, 100.0, 50.0
n_up = int((je_final - je_initial) / je_step)

steps = []
t = 0.0
for seg in [{'je_start': 0, 'je_end': je_final, 'n_steps': n_up},
            {'je_start': je_final, 'je_end': 0, 'n_steps': n_up}]:
    for i in range(seg['n_steps']):
        je_s = seg['je_start'] + i * je_step
        je_e = je_s + je_step
        steps.append({
            'je_start': je_s, 'je_end': je_e,
            'ramp_start': t, 'ramp_end': t + ramp_time,
            'stable_start': t + ramp_time, 'stable_end': t + ramp_time + stable_time,
            'save_start': t + ramp_time + stable_time,
            'save_end': t + ramp_time + stable_time + save_time,
        })
        t += ramp_time + stable_time + save_time
print(f'Steps: {len(steps)}, solve_time: {t:.0f}')

In [ ]:
class DummyTerminal:
    def __init__(self, pts):
        mask_x = pts[:, 0] < pts[:, 0].min() + 0.5
        self.site_indices = np.where(mask_x)[0]
class DummyDevice:
    def __init__(self, pts):
        self._pts = pts
        self._terminals = [DummyTerminal(pts), DummyTerminal(pts)]
    def terminal_info(self):
        return self._terminals
device = DummyDevice(points)
sim_done = threading.Event()
sim_done.set()
print('Ready')

In [ ]:
import ipywidgets as widgets
import h5py
from scipy.interpolate import griddata

import plotly.io as pio
pio.renderers.default = "jupyterlab"

# Check sim_done event (set by cell 22 background solver)
try:
    _ = sim_done
except NameError:
    sim_done = threading.Event()
    sim_done.set()

# Interpolation grid
xmin, xmax = points[:, 0].min(), points[:, 0].max()
ymin, ymax = points[:, 1].min(), points[:, 1].max()
nx, ny = 100, 50
gx = np.linspace(xmin, xmax, nx)
gy = np.linspace(ymin, ymax, ny)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])

# Read frame times from HDF5
def read_frame_times():
    if not os.path.exists(output_path):
        return np.array([])
    try:
        with h5py.File(output_path, 'r') as f:
            if 'data' not in f:
                return np.array([])
            total = len(f['data'].keys())
            if total == 0:
                return np.array([])
            return np.array([
                float(f[f'data/{fi}'].attrs.get('time', 0))
                for fi in range(total)
            ])
    except Exception:
        return np.array([])

# Compute heatmap for a given step index
def compute_frame(step_idx):
    s = steps[step_idx]
    frame_times = read_frame_times()
    if len(frame_times) == 0:
        return np.zeros((ny, nx)), {'je': s['je_end'], 'step': step_idx + 1, 'time': 0}

    mid_t = (s['save_start'] + s['save_end']) / 2
    mask = (frame_times >= s['save_start']) & (frame_times <= s['save_end'])
    step_fi = np.where(mask)[0]

    if len(step_fi) > 0:
        fi = step_fi[len(step_fi) // 2]
    else:
        fi = int(np.argmin(np.abs(frame_times - mid_t)))

    with h5py.File(output_path, 'r') as f:
        psi = np.abs(np.array(f[f'data/{fi}/psi']))

    Z = griddata(points, psi, grid_pts, method='cubic',
                 fill_value=0.0).reshape(ny, nx)
    Z = np.clip(Z, 0, None)
    info = {'je': s['je_end'], 'step': step_idx + 1,
            'time': (s['save_start'] + s['save_end']) / 2}
    return Z, info

n_steps = len(steps)

# Initial frame
initial_Z, initial_info = compute_frame(0)
print(f'Initial frame: shape={initial_Z.shape}, je={initial_info["je"]:.2f}')

# Electrode rectangles as shapes
electrode_shapes = []
for t in device.terminal_info():
    idx = t.site_indices
    x0, x1 = points[idx, 0].min(), points[idx, 0].max()
    y0, y1 = points[idx, 1].min(), points[idx, 1].max()
    electrode_shapes.append(dict(
        type='rect', x0=x0, x1=x1, y0=y0, y1=y1,
        line=dict(color='cyan', width=1.5, dash='dash'),
        fillcolor='rgba(0,0,0,0)', layer='above',
    ))

# Build Plotly FigureWidget
fig = go.FigureWidget(
    go.Heatmap(x=gx, y=gy, z=initial_Z,
               colorscale='Inferno', zmin=0, zmax=1.05,
               colorbar=dict(title='|psi|'),
               showscale=True),
)
fig.update_layout(
    title=f'|psi|  je={initial_info["je"]:.2f}  t={initial_info["time"]:.0f}s  '
          f'(step 1/{n_steps})',
    xaxis=dict(showline=True, linewidth=1, linecolor='black',
               mirror=True, ticks='outside'),
    yaxis=dict(scaleanchor='x', scaleratio=1, showline=True,
               linewidth=1, linecolor='black', mirror=True, ticks='outside'),
    margin=dict(l=50, r=10, t=35, b=50),
    height=400, width=700, plot_bgcolor='white',
    shapes=electrode_shapes,
)

# Play + Slider controls
slider = widgets.IntSlider(
    value=0, min=0, max=n_steps - 1, step=1,
    description='Step:', continuous_update=True,
)
play = widgets.Play(
    value=0, min=0, max=n_steps - 1, interval=500,
    description='Play',
)
widgets.jslink((play, 'value'), (slider, 'value'))

# Frame update callback
def on_slider_change(change):
    idx = change['new']
    Z, info = compute_frame(idx)
    fig.data[0].z = Z
    fig.update_layout(
        title=f'|psi|  je={info["je"]:.2f}  t={info["time"]:.0f}s  '
              f'(step {info["step"]}/{n_steps})')

slider.observe(on_slider_change, names='value')

# Display
controls = widgets.HBox([play, slider])
display(widgets.VBox([controls, fig]))
print(f'{n_steps} timing steps - use Play/Slider to browse frames')


In [ ]:
# Performance test: compute 5 frames
import time
for idx in [0, 50, 100, 150, 200]:
    if idx >= n_steps:
        continue
    t0 = time.time()
    Z, info = compute_frame(idx)
    elapsed = (time.time() - t0) * 1000
    print(f'  Step {idx+1}: je={info["je"]:.2f}, Z=[{Z.min():.3f}, {Z.max():.3f}], {elapsed:.0f}ms')
print('\nAll tests PASSED')